# CC18 flow-performance experiments

Runs the full experiment suite for the paper using the single `openml_flow` module.

1. Load CC18 tasks/evaluations (cached) and the referenced sklearn flows.
2. Run each representation config (TF-IDF / MiniLM / metafeatures-only) under task-group CV,
   computing SHAP-based explanation-quality metrics.
3. Consolidate per-run outputs into results-dir-level CSVs that `figures.ipynb` reads.

Edit only the **Configuration** cell. Everything else just calls `openml_flow`.

## Configuration

In [ ]:
from pathlib import Path

import pandas as pd

import openml_flow as wf

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# Paths
CACHE_DIR = "minimal_cache_cc18"
RESULTS_DIR_MAX = "results_cc18_max"
RESULTS_DIR_MEAN = "results_cc18_mean"
FLOWS_CACHE = Path(CACHE_DIR) / "flows.joblib"

# OpenML / CV settings
FUNCTION = "predictive_accuracy"
CV_FOLDS = 5
RANDOM_STATE = 42

# Restrict to the models used in the paper (None = all four).
SELECTED_MODELS = ["hist_gbrt", "random_forest"]

# SHAP settings (forwarded to run_cc18_pipeline).
SHAP_KWARGS = dict(
    compute_shap=True,
    shap_background_size=200,
    shap_eval_size=100,
    compute_shap_interactions=False,
    shap_interaction_eval_size=50,
    shap_top_k=10,
    shap_sparsity_mass=0.8,
)

# Representation configs. `name` becomes the run subdirectory.
MAX_CONFIGS = [
    {"name": "tfidf_meta", "text_mode": "tfidf", "use_task_metafeatures": True},
    {"name": "minilm_meta", "text_mode": "minilm", "use_task_metafeatures": True},
    {"name": "meta_only", "text_mode": "none", "use_task_metafeatures": True},
]
MEAN_CONFIGS = MAX_CONFIGS  # robustness runs under mean aggregation

## Load CC18 tasks and evaluations

In [ ]:
cache = wf.DiskCache(CACHE_DIR)

bundle = wf.load_cc18_from_openml(
    function=FUNCTION,
    suite_name="OpenML-CC18",
    cache=cache,
)
tasks_df = bundle["tasks_df"]
evals_df = bundle["evals_df"]

print("tasks_df:", tasks_df.shape, "| evals_df:", evals_df.shape)
tasks_df.head()

## Load (or cache) the referenced flows

In [ ]:
flows = wf.load_or_cache_flows(evals_df, FLOWS_CACHE)
len(flows)

## Run experiments

Task-group CV is the main protocol. Each config writes a run directory under the results dir
(`cv_results.csv`, `summary.csv`, SHAP CSVs, `shap_artifacts.pkl`, `run_metadata.json`).

In [ ]:
outputs_max = wf.run_experiment_configs(
    flows=flows,
    tasks_df=tasks_df,
    evals_df=evals_df,
    configs=MAX_CONFIGS,
    agg_mode="max",
    results_dir=RESULTS_DIR_MAX,
    selected_models=SELECTED_MODELS,
    cv_folds=CV_FOLDS,
    random_state=RANDOM_STATE,
    cache_dir=CACHE_DIR,
    **SHAP_KWARGS,
)
list(outputs_max)

In [ ]:
outputs_mean = wf.run_experiment_configs(
    flows=flows,
    tasks_df=tasks_df,
    evals_df=evals_df,
    configs=MEAN_CONFIGS,
    agg_mode="mean",
    results_dir=RESULTS_DIR_MEAN,
    selected_models=SELECTED_MODELS,
    cv_folds=CV_FOLDS,
    random_state=RANDOM_STATE,
    cache_dir=CACHE_DIR,
    **SHAP_KWARGS,
)
list(outputs_mean)

## Consolidate results for the figures notebook

In [ ]:
consolidated_max = wf.consolidate_experiments(outputs_max, RESULTS_DIR_MAX)
consolidated_mean = wf.consolidate_experiments(outputs_mean, RESULTS_DIR_MEAN)

print("Max-aggregation predictive anchor:")
consolidated_max["anchor"]